# `reproject_to_match` Minimal Same-Frame Walkthrough

This notebook demonstrates `reproject_to_match` for a same-frame case (FK5 -> FK5).

Setup:
- Input pixels are **1.4x larger** than template pixels.
- Input reference direction is offset by **(20.5, 20.5) input pixels**.
- Input image has **two Gaussian sources** in opposite quadrants.

Important clarification:
- The template image is used as a **coordinate template only**.
- Template intensities are set to **all zeros** by design.
- Scientific comparisons are input vs output, not template intensity vs output intensity.

## 1) Imports

In [ ]:
import numpy as np
import xarray as xr

from xradio.image import make_empty_sky_image
from wcs_reproject import build_wcs_from_xradio, reproject_to_match
from plotting import generate_astro_plot


## 2) Define geometry

In [ ]:
n_l = 192
n_m = 192

template_cell_arcsec = 1.5
input_cell_arcsec = 1.4 * template_cell_arcsec

cell_template = np.deg2rad(template_cell_arcsec / 3600.0)
cell_input = np.deg2rad(input_cell_arcsec / 3600.0)

# Required input-reference offset in input-pixel units.
ref_shift_l = 20.5 * cell_input
ref_shift_m = 20.5 * cell_input


## 3) Build input and template datasets (same FK5 frame)

In [ ]:
template = make_empty_sky_image(
    phase_center=[0.0, 0.0],
    image_size=[n_l, n_m],
    cell_size=[cell_template, cell_template],
    frequency_coords=np.array([1.4e9]),
    pol_coords=["I"],
    time_coords=np.array([59000.0]),
    direction_reference="fk5",
    projection="SIN",
    spectral_reference="lsrk",
    do_sky_coords=True,
)

input = make_empty_sky_image(
    phase_center=[ref_shift_l, ref_shift_m],
    image_size=[n_l, n_m],
    cell_size=[cell_input, cell_input],
    frequency_coords=np.array([1.4e9]),
    pol_coords=["I"],
    time_coords=np.array([59000.0]),
    direction_reference="fk5",
    projection="SIN",
    spectral_reference="lsrk",
    do_sky_coords=True,
)

print("template cell arcsec:", template_cell_arcsec)
print("input cell arcsec:", input_cell_arcsec)
print("input/template pixel scale ratio:", input_cell_arcsec / template_cell_arcsec)
print("reference shift in input pixels:", 20.5, 20.5)


## 3b) Prove reference-direction offset in input-pixel units

In [ ]:
input_ref = np.asarray(input.attrs["coordinate_system_info"]["reference_direction"]["data"], dtype=float)
template_ref = np.asarray(template.attrs["coordinate_system_info"]["reference_direction"]["data"], dtype=float)
delta_ref = input_ref - template_ref

input_dl = float(np.nanmedian(np.diff(input["l"].values)))
input_dm = float(np.nanmedian(np.diff(input["m"].values)))

# Signed pixel offsets follow axis handedness; for RA-like axis l, spacing is negative.
shift_l_pix_signed = float(delta_ref[0] / input_dl)
shift_m_pix_signed = float(delta_ref[1] / input_dm)

# Magnitudes are often what users mean by "offset by N pixels".
shift_l_pix_mag = float(abs(shift_l_pix_signed))
shift_m_pix_mag = float(abs(shift_m_pix_signed))

print("input reference direction (rad):", input_ref)
print("template reference direction (rad):", template_ref)
print("reference-direction delta (rad):", delta_ref)
print("input pixel size (rad/pixel):", (input_dl, input_dm))
print("derived signed shift in input pixels:", (shift_l_pix_signed, shift_m_pix_signed))
print("derived magnitude shift in input pixels:", (shift_l_pix_mag, shift_m_pix_mag))
print("note: negative l-shift sign is expected when l-axis spacing is negative (RA increases left).")

assert np.allclose([shift_l_pix_mag, shift_m_pix_mag], [20.5, 20.5], rtol=0.0, atol=1e-10)


## 3c) RA handedness note (why the plotted horizontal shift can look sign-flipped)

RA increases to the **left** in standard astronomical orientation.
So a positive offset in local direction-coordinate metadata (`l`) does not imply a positive
screen/pixel-x shift on the displayed RA axis.
The cell below computes this directly from WCS.

In [ ]:
input_ref = input.attrs["coordinate_system_info"]["reference_direction"]["data"]
input_ra_deg = float(np.rad2deg(input_ref[0]))
input_dec_deg = float(np.rad2deg(input_ref[1]))

template_wcs = build_wcs_from_xradio(template["SKY"] if "SKY" in template else template, dim_a="l", dim_b="m").wcs
ref_i, ref_j = template_wcs.wcs_world2pix(input_ra_deg, input_dec_deg, 0)
ref_i = float(np.asarray(ref_i).reshape(-1)[0])
ref_j = float(np.asarray(ref_j).reshape(-1)[0])
ctr = (n_l - 1) / 2.0
di = float(ref_i - ctr)
dj = float(ref_j - ctr)
cdelt = tuple(float(v) for v in template_wcs.wcs.cdelt)

print("input reference direction in template pixel coords:", (ref_i, ref_j))
print("delta from template image center (pixels):", (di, dj))
print("template CDELT (deg/pix):", cdelt)
print("Interpretation: CDELT1 < 0 => RA increases left, so horizontal sign is flipped in display coordinates.")


## 4) Populate input with two Gaussians (opposite quadrants, different sizes)

In [ ]:
input_components = [
    (48.0, 140.0, 3.8, 6.3, 1.0),
    (142.0, 52.0, 7.5, 2.9, 0.8),
]

def gaussian_field_pixel(n_i, n_j, components):
    ii, jj = np.indices((n_i, n_j), dtype=np.float64)
    out = np.zeros((n_i, n_j), dtype=np.float64)
    for i0, j0, sig_i, sig_j, amp in components:
        out += amp * np.exp(-0.5 * (((ii - i0) / sig_i) ** 2 + ((jj - j0) / sig_j) ** 2))
    return out

input_plane = gaussian_field_pixel(n_l, n_m, input_components)
input["SKY"] = xr.DataArray(
    input_plane[None, None, None, :, :],
    dims=("time", "frequency", "polarization", "l", "m"),
    coords={d: input.coords[d] for d in ("time", "frequency", "polarization", "l", "m")},
)
input["SKY"].attrs["units"] = "Jy/pixel"

# Plot input right after population.
input_wcs_now = build_wcs_from_xradio(input["SKY"], dim_a="l", dim_b="m").wcs
input_img_now = input["SKY"].isel(time=0, frequency=0, polarization=0).values
fig_input_now, ax_input_now = generate_astro_plot(input_img_now, input_wcs_now)
ax_input_now.set_title("Input image right after Gaussian population")
ax_input_now.coords[0].set_axislabel("Right Ascension")
ax_input_now.coords[1].set_axislabel("Declination")


## 5) Set template intensities to zero (coordinate-template only)

In [ ]:
# Template intensity is intentionally irrelevant for this workflow.
template["SKY"] = xr.DataArray(
    np.zeros((1, 1, 1, n_l, n_m), dtype=np.float64),
    dims=("time", "frequency", "polarization", "l", "m"),
    coords={d: template.coords[d] for d in ("time", "frequency", "polarization", "l", "m")},
)
template["SKY"].attrs["units"] = "Jy/pixel"


## 6) Reproject input onto template coordinates

In [ ]:
out = reproject_to_match(input, template, data_var="SKY", method="bilinear")
print("output SKY shape:", out["SKY"].shape)


## 7) Plot reprojected output image

In [ ]:
out_wcs = build_wcs_from_xradio(out["SKY"], dim_a="l", dim_b="m").wcs
out_img = out["SKY"].isel(time=0, frequency=0, polarization=0).values
fig_o, ax_o = generate_astro_plot(out_img, out_wcs)
ax_o.set_title("Reprojected output on template grid")
ax_o.coords[0].set_axislabel("Right Ascension")
ax_o.coords[1].set_axislabel("Declination")


## 8) Validation A: output world-coordinate grids equal template grids

In [ ]:
ra_equal = np.allclose(out["right_ascension"].values, template["right_ascension"].values, rtol=0.0, atol=0.0)
dec_equal = np.allclose(out["declination"].values, template["declination"].values, rtol=0.0, atol=0.0)
print("right_ascension grids identical:", ra_equal)
print("declination grids identical:", dec_equal)
assert ra_equal and dec_equal


## 9) Validation B: mapped input-peak vs output-peak pixel diagnostic (informational)

In [ ]:
# Validation B is a secondary diagnostic in output-pixel space.
# Pixel peaks need not match exactly under reprojection/resampling;
# world-coordinate agreement is the primary physical criterion.

def peak_in_window(img2d, center_i, center_j, half_width=16):
    i0 = max(int(round(center_i)) - half_width, 0)
    i1 = min(int(round(center_i)) + half_width + 1, img2d.shape[0])
    j0 = max(int(round(center_j)) - half_width, 0)
    j1 = min(int(round(center_j)) + half_width + 1, img2d.shape[1])
    sub = img2d[i0:i1, j0:j1]
    ii, jj = np.unravel_index(np.nanargmax(sub), sub.shape)
    return i0 + ii, j0 + jj

input_wcs = build_wcs_from_xradio(input["SKY"], dim_a="l", dim_b="m").wcs
out_wcs = build_wcs_from_xradio(out["SKY"], dim_a="l", dim_b="m").wcs
input_img = input["SKY"].isel(time=0, frequency=0, polarization=0).values
out_img = out["SKY"].isel(time=0, frequency=0, polarization=0).values

results = []
for k, (input_ci, input_cj, _, _, _) in enumerate(input_components, start=1):
    input_i, input_j = peak_in_window(input_img, input_ci, input_cj)
    input_i, input_j = int(input_i), int(input_j)

    input_ra_deg, input_dec_deg = input_wcs.wcs_pix2world(float(input_i), float(input_j), 0)
    map_i, map_j = out_wcs.wcs_world2pix(float(input_ra_deg), float(input_dec_deg), 0)
    map_i = float(np.asarray(map_i).reshape(-1)[0])
    map_j = float(np.asarray(map_j).reshape(-1)[0])

    out_i, out_j = peak_in_window(out_img, map_i, map_j)
    out_i, out_j = int(out_i), int(out_j)
    dpix = float(np.hypot(out_i - map_i, out_j - map_j))
    results.append((k, (input_i, input_j), (map_i, map_j), (out_i, out_j), dpix))

print("Pixel Coordinates")
header = ("source", "input peak", "output peak", "predicted peak")
rows = []
for k, input_pk, mapped, out_pk, d in results:
    predicted = f"({mapped[0]:.3f}, {mapped[1]:.3f})"
    rows.append((str(k), str(input_pk), str(out_pk), predicted))

widths = [len(h) for h in header]
for r in rows:
    for i, v in enumerate(r):
        widths[i] = max(widths[i], len(v))

def fmt_row(vals):
    return "| " + " | ".join(v.ljust(widths[i]) for i, v in enumerate(vals)) + " |"

print(fmt_row(header))
print("| " + " | ".join("-" * w for w in widths) + " |")
for r in rows:
    print(fmt_row(r))

max_offset = max(r[-1] for r in results)
print("\nmax mapped-peak pixel offset:", f"{max_offset:.3f}")
print("note: this is a pixel-grid diagnostic only; world-coordinate preservation is validated in section 10.")


## 10) Validation C: input/output peak world coordinates agree

In [ ]:
input_wcs = build_wcs_from_xradio(input["SKY"], dim_a="l", dim_b="m").wcs
out_wcs = build_wcs_from_xradio(out["SKY"], dim_a="l", dim_b="m").wcs
input_img = input["SKY"].isel(time=0, frequency=0, polarization=0).values
out_img = out["SKY"].isel(time=0, frequency=0, polarization=0).values

# Tolerance: half the larger pixel size between input and template.
world_tol_arcsec = 0.5 * max(input_cell_arcsec, template_cell_arcsec)
print("world-coordinate peak-match tolerance (arcsec):", world_tol_arcsec)

def world_sep_arcsec(ra1, dec1, ra2, dec2):
    d_ra = np.arctan2(np.sin(ra2 - ra1), np.cos(ra2 - ra1))
    mean_dec = 0.5 * (dec1 + dec2)
    d_lon = d_ra * np.cos(mean_dec)
    d_lat = dec2 - dec1
    return float(np.rad2deg(np.hypot(d_lon, d_lat)) * 3600.0)

seps = []
for k, (input_ci, input_cj, _, _, _) in enumerate(input_components, start=1):
    input_i, input_j = peak_in_window(input_img, input_ci, input_cj)
    input_ra_deg, input_dec_deg = input_wcs.wcs_pix2world(float(input_i), float(input_j), 0)
    map_i, map_j = out_wcs.wcs_world2pix(float(input_ra_deg), float(input_dec_deg), 0)
    map_i = float(np.asarray(map_i).reshape(-1)[0])
    map_j = float(np.asarray(map_j).reshape(-1)[0])
    out_i, out_j = peak_in_window(out_img, map_i, map_j)
    out_ra_deg, out_dec_deg = out_wcs.wcs_pix2world(float(out_i), float(out_j), 0)

    input_ra = float(np.deg2rad(input_ra_deg)); input_dec = float(np.deg2rad(input_dec_deg))
    out_ra = float(np.deg2rad(out_ra_deg)); out_dec = float(np.deg2rad(out_dec_deg))
    sep = world_sep_arcsec(input_ra, input_dec, out_ra, out_dec)
    seps.append(sep)
    print(f"source {k}:")
    print(f"  Input    RA/Dec (deg): ({input_ra_deg:11.8f}, {input_dec_deg:11.8f})")
    print(f"  Output   RA/Dec (deg): ({out_ra_deg:11.8f}, {out_dec_deg:11.8f})")
    print(f"  sep_arcsec         : {sep:7.4f}")
    print()

max_sep = max(seps)
print("max input/output world-coordinate peak separation (arcsec):", max_sep)
assert max_sep <= world_tol_arcsec
